# **ML_2. Supervised Learning**

### **1. Ответь на вопросы**

*i. Выведите аналитическое решение задачи регрессии. Используйте векторную форму уравнения.*   

**ii. Какие изменения произойдут в решении при добавлении L1- и L2-регуляризации к функции потерь?*  

*iii. Объясните, почему L1-регуляризация часто используется для отбора признаков. Почему после обучения модели остается много весов, равных 0?*    

*iV. Объясните, как можно использовать те же модели (линейная регрессия, гребневая регрессия и т. д.), но при этом обеспечить возможность учета нелинейных зависимостей*  

### **2. Введение**.  
*Выполните всю предварительную обработку данных из предыдущего урока.*

#### ***i. Импорт библиотек***

In [79]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

#### ***ii. Загрузка train и test датасетов. (read train and test parts)***

In [80]:
data_train = pd.read_json("../datasets/train.json")
print(f"""Количество строк: {data_train.shape[0]}
Количество столбцов: {data_train.shape[1]}""")

Количество строк: 49352
Количество столбцов: 15


In [81]:
data_test = pd.read_json("../datasets/test.json")
print(f"""Количество строк: {data_test.shape[0]}
Количество столбцов: {data_test.shape[1]}""")

Количество строк: 74659
Количество столбцов: 14


### **3. Введение в анализ данных, часть 2**.  

#### ***i. Давайте создадим дополнительные характеристики для повышения качества модели. Рассмотрим столбец под названием «Features». Он содержит список основных особенностей текущей квартиры.***

In [82]:
cols = ["bathrooms", "bedrooms", "features"]
data_train[cols].head(10)

,bathrooms,bedrooms,features
4,1.0,1,"[Dining Room, Pre-War, Laundry in Building, Di..."
6,1.0,2,"[Doorman, Elevator, Laundry in Building, Dishw..."
9,1.0,2,"[Doorman, Elevator, Laundry in Building, Laund..."
10,1.5,3,[]
15,1.0,0,"[Doorman, Elevator, Fitness Center, Laundry in..."
16,1.0,3,"[Doorman, Elevator, Loft, Dishwasher, Hardwood..."
18,2.0,3,"[Fireplace, Laundry in Unit, Dishwasher, Hardw..."
19,1.0,0,"[Elevator, Laundry in Building, Dishwasher, Ha..."
23,0.0,1,[Hardwood Floors]
32,3.0,3,"[Cats Allowed, Dogs Allowed]"


#### ***ii. Удалите из столбца неиспользуемые символы ([,], ', ", и пробел).***

*после pd.read_json столбец features это не строка, а список Python. Поэтому для применения строковых методов нужно привести к строке.*

In [83]:
print(type(data_train["features"].iloc[0]))

<class 'list'>


In [84]:
data_train["features"].head(3)

4    [Dining Room, Pre-War, Laundry in Building, Di...
6    [Doorman, Elevator, Laundry in Building, Dishw...
9    [Doorman, Elevator, Laundry in Building, Laund...
Name: features, dtype: object

In [85]:
data_train["features"] = data_train["features"].astype(str)
data_train["features"].iloc[0]

"['Dining Room', 'Pre-War', 'Laundry in Building', 'Dishwasher', 'Hardwood Floors', 'Dogs Allowed', 'Cats Allowed']"

In [86]:
print(type(data_train["features"].iloc[0]))

<class 'str'>


In [87]:
for ch in ["[", "]", "'", '"', " "]:
    data_train["features"] = data_train["features"].str.replace(ch, "", regex=False)

data_train["features"].head(3)

4    DiningRoom,Pre-War,LaundryinBuilding,Dishwashe...
6    Doorman,Elevator,LaundryinBuilding,Dishwasher,...
9    Doorman,Elevator,LaundryinBuilding,LaundryinUn...
Name: features, dtype: str

In [ ]:
print(
    "Всего пустых строк  в data_train датасете:", (data_train["features"] == "").sum()
)

Всего пустых строк  в data_train датасете: 3218


#### ***iii. Получите все значения в каждом списке и соберите результат в один большой список для всего набора данных. Для этого можно использовать DataFrame.iterrows().***

In [92]:
data_train["features"].iloc[0].split(",")

['DiningRoom',
 'Pre-War',
 'LaundryinBuilding',
 'Dishwasher',
 'HardwoodFloors',
 'DogsAllowed',
 'CatsAllowed']

In [ ]:
huge_list = []
for _, row in data_train.iterrows():
    if row["features"] != "":
        huge_list.extend(row["features"].split(","))

print("Длина огромного списка:", len(huge_list))
print("Первые 10 элементов:", huge_list[:10])

Длина огромного списка: 267906
Первые 10 элементов: ['DiningRoom', 'Pre-War', 'LaundryinBuilding', 'Dishwasher', 'HardwoodFloors', 'DogsAllowed', 'CatsAllowed', 'Doorman', 'Elevator', 'LaundryinBuilding']
Пустых элементов внутри: 0


#### ***iv. Сколько уникальных значений содержит список результатов?***